# 📖 Notebook 3: Ledger & Double-Entry Bookkeeping

In a payment system, every dollar must be accounted for. If your numbers don't add up, you have a bug — or fraud.  
The solution used by every financial system since the 15th century is **double-entry bookkeeping**.

## The Core Rule

Every time money moves, we record **two** entries:
- A **debit** (money coming from somewhere)
- A **credit** (money going to somewhere)

The total of all debits must **always** equal the total of all credits. If they don't, something is wrong.

## Learning Objectives

By the end of this notebook you'll understand:
- Why double-entry bookkeeping exists and how it catches errors
- How to model ledger entries in a database
- How to verify the books balance (debits == credits)
- How to query account balances and transaction history

## 🛠️ Setup

```bash
cd 06-system-designs/payment-system
docker compose up -d
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import psycopg2.extras
import uuid

DB_CONFIG = {
    "host": "localhost", "port": 5432,
    "database": "payment_demo", "user": "demo", "password": "demo"
}

def get_db():
    return psycopg2.connect(**DB_CONFIG)

conn = get_db()
conn.autocommit = True
cur = conn.cursor()
# Section 8 installs a constraint trigger that makes an unbalanced ledger
# impossible. Section 7 deliberately creates one to show what detection looks
# like, so start every run from the plain schema and let section 8 add it back.
cur.execute("DROP TRIGGER IF EXISTS trg_ledger_balanced ON ledger_entries")
cur.close()
conn.close()
print("✅ Postgres connected")

---
## 1. Why Double-Entry? A Simple Analogy

Think of it like moving a ball between two boxes:

- **Box A** (the customer's wallet) loses the ball → that's a **debit** on their account.
- **Box B** (the merchant's account) gains the ball → that's a **credit** on their account.

If you ever count all the balls and find the total has changed, something went wrong.  
Double-entry bookkeeping is just a formal way of making sure **no balls disappear or appear from nowhere**.

### Our Accounts

For a simplified payment system, we use two main accounts:

| Account | What It Represents |
|---------|-------------------|
| `customer_receivable` | Money we expect to collect from the customer's card |
| `merchant_payable` | Money we owe to the merchant |

When a charge succeeds:
- **Debit** `customer_receivable` (we received money from the customer)
- **Credit** `merchant_payable` (we owe that money to the merchant)

---
## 2. Bad Practice First: Single-Entry Bookkeeping

Before we admire double-entry, let's see what happens **without** it.

Naive systems just record `merchant X earned $Y` in a single `earnings` table. One row, one number. Simple, right? The problem:

- If the row is wrong (typo, bug, malicious edit), there is **nothing to compare it to**.
- You cannot tell where the money came from or where it went -- there is no `other side`.
- You cannot detect partial writes: if the app crashes mid-save, the single number just ends up wrong with no alarm.

Let's build a tiny single-entry version and see why it breaks.


In [ ]:
# BAD: single-entry bookkeeping -- one row per payment, no counterpart.
# We store everything in a throwaway temp table so we don't pollute our real schema.
conn = get_db()
cur = conn.cursor()
cur.execute("CREATE TEMP TABLE IF NOT EXISTS earnings_single_entry (merchant_id TEXT, amount_cents INTEGER)")
cur.execute("TRUNCATE earnings_single_entry")

# Record 3 legitimate payments...
for merchant, amount in [("merch_001", 5000), ("merch_002", 2500), ("merch_001", 1000)]:
    cur.execute("INSERT INTO earnings_single_entry VALUES (%s, %s)", (merchant, amount))

# ...and now simulate a bug: one row gets written with the wrong amount (off-by-one-zero).
cur.execute("INSERT INTO earnings_single_entry VALUES ('merch_002', 99999)")  # should have been 9999

cur.execute("SELECT merchant_id, SUM(amount_cents) FROM earnings_single_entry GROUP BY merchant_id ORDER BY merchant_id")
print("Single-entry 'earnings' table:")
for row in cur.fetchall():
    print(f"  {row[0]}: ${row[1]/100:.2f}")

print()
print("Problem: nothing in this schema can tell us row #4 is wrong.")
print("There is no counterpart entry to compare against. The books 'balance'")
print("against nothing, because there is no other side.")
cur.close(); conn.close()


---
## 3. Examining the Existing Ledger

Our `init.sql` already seeded some ledger entries. Let's look at them.

In [ ]:
conn = get_db()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

# Show all ledger entries
cur.execute("""
    SELECT le.transaction_id, le.account_name, le.entry_type, 
           le.amount_cents, le.description
    FROM ledger_entries le
    ORDER BY le.transaction_id, le.entry_type
""")

rows = cur.fetchall()
print(f"Total ledger entries: {len(rows)}\n")
print(f"{'Transaction':<15} {'Account':<25} {'Type':<8} {'Amount':>10}  Description")
print("-" * 90)
for r in rows:
    amt = f"${r['amount_cents']/100:.2f}"
    print(f"{r['transaction_id']:<15} {r['account_name']:<25} {r['entry_type']:<8} {amt:>10}  {r['description']}")

cur.close(); conn.close()

Notice the pattern: **every transaction has exactly two entries** — one debit and one credit for the same amount.

---
## 4. The Balance Check: Do the Books Balance?

The fundamental accounting equation:  
**Total Debits = Total Credits**

If this ever fails, we have a bug. Let's verify.

In [ ]:
def check_books_balance(strict=True):
    """Verify that total debits equal total credits across all ledger entries.

    `strict=True` raises instead of printing a sad face. In a payment system a
    failed balance check is not a log line, it is a page: you stop settling
    money until a human has looked at it.

    ⚠️  Scaling note: this SUMs the ENTIRE ledger. Notebook 1's estimate put
    that at ~1.4 trillion rows over a 7-year retention window — hours per run.
    Real systems close the books daily (store a signed per-account balance per
    day) and verify only today against yesterday's close. The full scan then
    becomes a monthly job on a read replica, kept as the only check that can
    catch a corrupt baseline.
    """
    conn = get_db()
    cur = conn.cursor()

    cur.execute("""
        SELECT 
            SUM(CASE WHEN entry_type = 'debit'  THEN amount_cents ELSE 0 END) AS total_debits,
            SUM(CASE WHEN entry_type = 'credit' THEN amount_cents ELSE 0 END) AS total_credits
        FROM ledger_entries
    """)
    row = cur.fetchone()
    total_debits = row[0] or 0
    total_credits = row[1] or 0

    print(f"Total Debits  : ${total_debits/100:.2f}")
    print(f"Total Credits : ${total_credits/100:.2f}")
    print(f"Difference    : ${(total_debits - total_credits)/100:.2f}")

    balanced = total_debits == total_credits
    if balanced:
        print("\n✅ Books balance! Every dollar is accounted for.")
    else:
        print("\n🚨 BOOKS DO NOT BALANCE! Something is wrong!")

    cur.close(); conn.close()
    if strict and not balanced:
        raise AssertionError(
            f"ledger out of balance by {total_debits - total_credits} cents")
    return balanced

check_books_balance()

---
## 5. Recording a New Charge with Ledger Entries

Let's build a function that processes a charge and records both ledger entries in a single database transaction.  
Using a database transaction (`BEGIN ... COMMIT`) ensures that either **both** entries are written, or **neither** is. We never end up with a debit without its matching credit.

In [ ]:
def record_charge_with_ledger(merchant_id, amount_cents, description):
    """
    Create a PaymentIntent, a Transaction, and the matching ledger entries.
    Everything happens in a single database transaction for atomicity.
    """
    pi_id = f"pi_{uuid.uuid4().hex[:12]}"
    txn_id = f"txn_{uuid.uuid4().hex[:12]}"
    conn = get_db()
    cur = conn.cursor()

    try:
        # Create PaymentIntent (succeeded immediately for this demo)
        cur.execute("""
            INSERT INTO payment_intents (id, merchant_id, amount_cents, currency, description, status)
            VALUES (%s, %s, %s, 'usd', %s, 'succeeded')
        """, (pi_id, merchant_id, amount_cents, description))

        # Create Transaction
        cur.execute("""
            INSERT INTO transactions (id, payment_intent_id, type, amount_cents, currency, status, card_last_four, card_brand)
            VALUES (%s, %s, 'charge', %s, 'usd', 'succeeded', '4242', 'visa')
        """, (txn_id, pi_id, amount_cents))

        # Double-entry ledger: DEBIT customer_receivable, CREDIT merchant_payable
        cur.execute("""
            INSERT INTO ledger_entries (transaction_id, account_name, entry_type, amount_cents, currency, description)
            VALUES
                (%s, 'customer_receivable', 'debit',  %s, 'usd', %s),
                (%s, 'merchant_payable',    'credit', %s, 'usd', %s)
        """, (txn_id, amount_cents, f"Charge for {pi_id}",
              txn_id, amount_cents, f"Charge for {pi_id}"))

        conn.commit()
        print(f"✅ Charge recorded:")
        print(f"   PaymentIntent : {pi_id}")
        print(f"   Transaction   : {txn_id}")
        print(f"   Amount        : ${amount_cents/100:.2f}")
        print(f"   Ledger        : DEBIT customer_receivable ${amount_cents/100:.2f}")
        print(f"                   CREDIT merchant_payable   ${amount_cents/100:.2f}")
        return txn_id

    except Exception as e:
        conn.rollback()
        print(f"❌ Error: {e}")
        return None

    finally:
        cur.close(); conn.close()

# Record a few charges
record_charge_with_ledger("merch_001", 5999, "Ledger demo: premium headphones")
print()
record_charge_with_ledger("merch_002", 1250, "Ledger demo: basic widget")

In [ ]:
# Verify the books still balance after our new charges
check_books_balance()

---
## 6. Account Balances

We can compute the balance of any account by summing its debits and credits.  
This is how you'd answer questions like "how much do we owe merchant X?" or "how much have we collected from customers?"

In [ ]:
def show_account_balances():
    """Show the balance of each account in the ledger."""
    conn = get_db()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

    cur.execute("""
        SELECT 
            account_name,
            SUM(CASE WHEN entry_type = 'debit'  THEN amount_cents ELSE 0 END) AS total_debits,
            SUM(CASE WHEN entry_type = 'credit' THEN amount_cents ELSE 0 END) AS total_credits,
            COUNT(*) AS entry_count
        FROM ledger_entries
        GROUP BY account_name
        ORDER BY account_name
    """)

    print(f"{'Account':<25} {'Debits':>12} {'Credits':>12} {'Net':>12}  Entries")
    print("-" * 75)
    for r in cur.fetchall():
        net = r['total_debits'] - r['total_credits']
        print(f"{r['account_name']:<25} ${r['total_debits']/100:>10.2f} ${r['total_credits']/100:>10.2f} ${net/100:>10.2f}  {r['entry_count']}")

    cur.close(); conn.close()

show_account_balances()

---
## 7. What Happens When the Books Don't Balance?

Let's intentionally break the ledger by inserting a single entry without its matching pair.  
This simulates a bug where the credit entry wasn't written (e.g., the app crashed between the two inserts).

In [ ]:
# Simulate a bug: the app creates a Transaction and writes the DEBIT entry,
# then crashes BEFORE writing the matching CREDIT. Without a safety net the
# ledger ends up with an orphan debit.
import uuid
bug_pi_id  = f"pi_{uuid.uuid4().hex[:12]}"
bug_txn_id = f"txn_{uuid.uuid4().hex[:12]}"

conn = get_db()
cur = conn.cursor()
# Seed a real payment_intent + transaction so the FK is satisfied (this part
# of the code "worked"). The missing credit is what simulates the crash.
cur.execute("""
    INSERT INTO payment_intents (id, merchant_id, amount_cents, currency, description, status)
    VALUES (%s, 'merch_001', 9999, 'usd', 'BUG demo: app crashed mid-write', 'succeeded')
""", (bug_pi_id,))
cur.execute("""
    INSERT INTO transactions (id, payment_intent_id, type, amount_cents, currency, status, card_last_four, card_brand)
    VALUES (%s, %s, 'charge', 9999, 'usd', 'succeeded', '4242', 'visa')
""", (bug_txn_id, bug_pi_id))

# Now write ONLY the debit -- the credit is "lost" to the crash
cur.execute("""
    INSERT INTO ledger_entries (transaction_id, account_name, entry_type, amount_cents, currency, description)
    VALUES (%s, 'customer_receivable', 'debit', 9999, 'usd', 'BUG: missing credit entry')
""", (bug_txn_id,))
conn.commit()
cur.close(); conn.close()

print(f"Inserted a debit on {bug_txn_id} with NO matching credit...")
print()
# strict=False: this cell is SUPPOSED to be broken, so we want the report
# rather than the exception.
assert check_books_balance(strict=False) is False, "expected the books to be broken here"


In [ ]:
# Find the broken entry
conn = get_db()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

# Find transactions where debits != credits
cur.execute("""
    SELECT 
        transaction_id,
        SUM(CASE WHEN entry_type = 'debit'  THEN amount_cents ELSE 0 END) AS debits,
        SUM(CASE WHEN entry_type = 'credit' THEN amount_cents ELSE 0 END) AS credits
    FROM ledger_entries
    GROUP BY transaction_id
    HAVING SUM(CASE WHEN entry_type = 'debit' THEN amount_cents ELSE 0 END)
        != SUM(CASE WHEN entry_type = 'credit' THEN amount_cents ELSE 0 END)
""")

broken = cur.fetchall()
print("🔍 Transactions where debits ≠ credits:")
for r in broken:
    print(f"  {r['transaction_id']}: debits=${r['debits']/100:.2f}  credits=${r['credits']/100:.2f}")

cur.close(); conn.close()

In [ ]:
# Clean up the broken entry, and the transaction/intent we created for the demo.
# Order matters because of foreign keys: ledger_entries -> transactions -> payment_intents.
conn = get_db()
cur = conn.cursor()
cur.execute("DELETE FROM ledger_entries WHERE transaction_id = %s", (bug_txn_id,))
cur.execute("DELETE FROM transactions    WHERE id = %s", (bug_txn_id,))
cur.execute("DELETE FROM payment_intents WHERE id = %s", (bug_pi_id,))
conn.commit()
cur.close(); conn.close()

print("Cleaned up the broken entry and its parent rows.")
check_books_balance()


---
## 8. Making an Unbalanced Ledger *Impossible*

We just detected an orphan debit after the fact. Detection is necessary, but on
its own it means your ledger can be wrong for however long it takes the next
check to run — and during that window you are settling money against numbers
you cannot vouch for.

The stronger position: make the database **refuse to commit** an unbalanced
transaction. Postgres can do this with a `CONSTRAINT TRIGGER` declared
`DEFERRABLE INITIALLY DEFERRED`, which fires at `COMMIT` rather than after each
statement. That timing is the whole trick — a per-statement check would fire
after the debit and reject a perfectly good two-statement insert.

```
BEGIN
  INSERT debit    →  no check yet (deferred)
  INSERT credit   →  no check yet
COMMIT            →  trigger fires: does this transaction balance?
                     no  → the whole transaction is rejected
```

**What it costs you:**

- **Every commit that touches the ledger pays for an aggregate query** over the
  affected transaction's rows. Small here (2–4 rows), but it is not free, and
  it runs while your commit is holding locks.
- **It only checks the transactions you touched**, so it cannot catch a
  historical imbalance introduced before the trigger existed. You still need
  the periodic full check.
- **`DELETE` becomes dangerous** — removing one side of a pair now fails at
  commit, which is correct but will surprise anyone writing a cleanup script.
  (That's a feature.)

In [ ]:
conn = get_db()
conn.autocommit = True
cur = conn.cursor()

cur.execute("""
    CREATE OR REPLACE FUNCTION assert_ledger_balanced() RETURNS TRIGGER AS $$
    DECLARE
        txn TEXT := COALESCE(NEW.transaction_id, OLD.transaction_id);
        diff BIGINT;
    BEGIN
        SELECT COALESCE(SUM(CASE WHEN entry_type = 'debit'  THEN amount_cents
                                 ELSE -amount_cents END), 0)
          INTO diff
          FROM ledger_entries
         WHERE transaction_id = txn;

        -- Zero rows is fine (everything for this txn was deleted).
        IF diff <> 0 THEN
            RAISE EXCEPTION
                'ledger for transaction % is out of balance by % cents', txn, diff;
        END IF;
        RETURN NULL;
    END;
    $$ LANGUAGE plpgsql;
""")

cur.execute("DROP TRIGGER IF EXISTS trg_ledger_balanced ON ledger_entries")
cur.execute("""
    CREATE CONSTRAINT TRIGGER trg_ledger_balanced
    AFTER INSERT OR UPDATE OR DELETE ON ledger_entries
    DEFERRABLE INITIALLY DEFERRED
    FOR EACH ROW EXECUTE FUNCTION assert_ledger_balanced();
""")
conn.close()
print("🔒 Deferred constraint trigger installed on ledger_entries\n")


def try_ledger_write(label, statements):
    """Run some ledger statements in one transaction and report the outcome."""
    conn = get_db()
    cur = conn.cursor()
    try:
        for sql, params in statements:
            cur.execute(sql, params)
        conn.commit()
        print(f"  {label:<44} → COMMITTED")
        return True
    except psycopg2.errors.RaiseException as e:
        conn.rollback()
        msg = str(e).strip().splitlines()[0]
        print(f"  {label:<44} → REJECTED  ({msg})")
        return False
    finally:
        cur.close(); conn.close()


# Scaffolding: a real intent + transaction so the FK is satisfied.
def scaffold(amount=4200):
    pi = f"pi_{uuid.uuid4().hex[:12]}"
    txn = f"txn_{uuid.uuid4().hex[:12]}"
    conn = get_db(); cur = conn.cursor()
    cur.execute("""INSERT INTO payment_intents (id, merchant_id, amount_cents, currency, description, status)
                   VALUES (%s, 'merch_001', %s, 'usd', 'trigger demo', 'succeeded')""", (pi, amount))
    cur.execute("""INSERT INTO transactions (id, payment_intent_id, type, amount_cents, currency, status)
                   VALUES (%s, %s, 'charge', %s, 'usd', 'succeeded')""", (txn, pi, amount))
    conn.commit(); cur.close(); conn.close()
    return pi, txn


INS = ("INSERT INTO ledger_entries (transaction_id, account_name, entry_type,"
       " amount_cents, currency, description) VALUES (%s, %s, %s, %s, 'usd', %s)")

print("Attempting three ledger writes:")
scaffolded = []

# 1. Balanced pair — must commit.
pi1, t1 = scaffold(); scaffolded.append((pi1, t1))
ok1 = try_ledger_write("balanced pair (debit 42.00 / credit 42.00)", [
    (INS, (t1, "customer_receivable", "debit", 4200, "ok")),
    (INS, (t1, "merchant_payable", "credit", 4200, "ok")),
])

# 2. Orphan debit — the exact bug from section 7. Must be rejected.
pi2, t2 = scaffold(); scaffolded.append((pi2, t2))
ok2 = try_ledger_write("orphan debit, credit lost to a 'crash'", [
    (INS, (t2, "customer_receivable", "debit", 4200, "missing credit")),
])

# 3. Mismatched amounts — a fat-fingered constant. Must be rejected.
pi3, t3 = scaffold(); scaffolded.append((pi3, t3))
ok3 = try_ledger_write("mismatched pair (debit 42.00 / credit 4.20)", [
    (INS, (t3, "customer_receivable", "debit", 4200, "typo")),
    (INS, (t3, "merchant_payable", "credit", 420, "typo")),
])

assert ok1 and not ok2 and not ok3, (ok1, ok2, ok3)
print()
print("✅ The balanced write committed; both broken writes were refused at COMMIT.")
print("   The orphan debit from section 7 can no longer be created at all.")
print()
check_books_balance()

# Tidy up the scaffolding. Delete ledger rows first, and note that deleting
# BOTH sides together is fine — the trigger only objects to leaving a remainder.
conn = get_db(); cur = conn.cursor()
for pi, txn in scaffolded:
    cur.execute("DELETE FROM ledger_entries WHERE transaction_id = %s", (txn,))
    cur.execute("DELETE FROM transactions WHERE id = %s", (txn,))
    cur.execute("DELETE FROM payment_intents WHERE id = %s", (pi,))
conn.commit(); cur.close(); conn.close()
print("\n🧹 Cleaned up the trigger-demo rows.")

---
## 8. Per-Merchant Balances

In a real system, you'd want to see how much each merchant has earned.  
We can join the ledger with transactions and payment intents to break down balances by merchant.

In [ ]:
conn = get_db()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

cur.execute("""
    SELECT 
        m.name AS merchant_name,
        COUNT(DISTINCT le.transaction_id) AS transaction_count,
        SUM(CASE WHEN le.entry_type = 'credit' AND le.account_name = 'merchant_payable' 
            THEN le.amount_cents ELSE 0 END) AS total_earned_cents
    FROM ledger_entries le
    JOIN transactions t ON le.transaction_id = t.id
    JOIN payment_intents pi ON t.payment_intent_id = pi.id
    JOIN merchants m ON pi.merchant_id = m.id
    GROUP BY m.name
    ORDER BY total_earned_cents DESC
""")

print(f"{'Merchant':<25} {'Transactions':>15} {'Total Earned':>15}")
print("-" * 60)
for r in cur.fetchall():
    print(f"{r['merchant_name']:<25} {r['transaction_count']:>15} ${r['total_earned_cents']/100:>13.2f}")

cur.close(); conn.close()

---
## 8. Refunds: Reversing the Ledger

Real payment systems need refunds. In double-entry land, a refund does not delete the original charge -- the ledger is **append-only**. Instead, we write **two new rows** that reverse the original: DEBIT `merchant_payable`, CREDIT `customer_receivable`.

After the refund, both accounts net to zero for that transaction, and anyone auditing the books can still see the full history (charge + refund), not just the final result.


In [ ]:
def record_refund(original_txn_id):
    """
    Issue a full refund by writing a reversing pair of ledger entries.
    We never modify or delete the original rows (append-only ledger).
    """
    refund_txn_id = f"txn_{uuid.uuid4().hex[:12]}"
    conn = get_db()
    cur = conn.cursor()
    try:
        cur.execute("""
            SELECT payment_intent_id, amount_cents, currency
            FROM transactions WHERE id = %s
        """, (original_txn_id,))
        row = cur.fetchone()
        if not row:
            print(f"Transaction {original_txn_id} not found"); return None
        pi_id, amount_cents, currency = row

        cur.execute("""
            INSERT INTO transactions (id, payment_intent_id, type, amount_cents, currency, status, card_last_four, card_brand)
            VALUES (%s, %s, 'refund', %s, %s, 'succeeded', '4242', 'visa')
        """, (refund_txn_id, pi_id, amount_cents, currency))

        # Reversing ledger entries: same accounts, debit <-> credit swapped
        cur.execute("""
            INSERT INTO ledger_entries (transaction_id, account_name, entry_type, amount_cents, currency, description)
            VALUES
                (%s, 'merchant_payable',    'debit',  %s, %s, %s),
                (%s, 'customer_receivable', 'credit', %s, %s, %s)
        """, (refund_txn_id, amount_cents, currency, f"Refund of {original_txn_id}",
              refund_txn_id, amount_cents, currency, f"Refund of {original_txn_id}"))
        conn.commit()
        print("Refund recorded:")
        print(f"   Original      : {original_txn_id}  (${amount_cents/100:.2f})")
        print(f"   Refund txn    : {refund_txn_id}")
        print(f"   Ledger        : DEBIT  merchant_payable    ${amount_cents/100:.2f}")
        print(f"                   CREDIT customer_receivable ${amount_cents/100:.2f}")
        return refund_txn_id
    except Exception as e:
        conn.rollback(); print(f"Error: {e}"); return None
    finally:
        cur.close(); conn.close()

# Refund one of our seeded charges (txn_001 = $49.99)
record_refund("txn_001")
print()
check_books_balance()


In [ ]:
# Zoom in on txn_001 + its refund: the charge and reversal together should net
# to ZERO on every account touched. That is the property auditors check.
conn = get_db()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
cur.execute("""
    SELECT le.transaction_id, t.type, le.account_name, le.entry_type, le.amount_cents
    FROM ledger_entries le
    JOIN transactions t ON t.id = le.transaction_id
    WHERE t.payment_intent_id = 'pi_001'
    ORDER BY le.transaction_id, le.entry_type
""")
for r in cur.fetchall():
    sign = "+" if r['entry_type']=='debit' else "-"
    print(f"  {r['transaction_id']:<16} {r['type']:<7} {r['account_name']:<22} {r['entry_type']:<6} {sign}${r['amount_cents']/100:.2f}")

cur.execute("""
    SELECT le.account_name,
           SUM(CASE WHEN le.entry_type='debit'  THEN le.amount_cents ELSE 0 END)
         - SUM(CASE WHEN le.entry_type='credit' THEN le.amount_cents ELSE 0 END) AS net_cents
    FROM ledger_entries le
    JOIN transactions t ON t.id = le.transaction_id
    WHERE t.payment_intent_id = 'pi_001'
    GROUP BY le.account_name
""")
print()
print("Net per account for pi_001 (should be $0.00 after refund):")
for r in cur.fetchall():
    print(f"  {r['account_name']:<22} ${r['net_cents']/100:.2f}")
cur.close(); conn.close()


---
## 10. Reconciliation: Your Books vs. Everyone Else's

An internally consistent ledger is necessary and **not sufficient**. Debits can
equal credits while your books disagree with reality in three different ways:

| Break | What it looks like | What it usually means |
|---|---|---|
| **Missing on our side** | The processor settled a charge we have no ledger entry for | Our worker crashed after authorising and before writing the ledger |
| **Missing on their side** | We booked a charge the processor never settled | We recorded an authorisation as if it were a capture |
| **Amount mismatch** | Both have it, different numbers | Currency conversion, partial capture, or a fee we didn't model |

Reconciliation is the daily job that pulls the processor's settlement file,
compares it line by line against our ledger, and produces a list of **breaks**
for a human. It is boring, it is unglamorous, and it is the single control that
catches everything the code missed.

Notice the first break type is not hypothetical in *this* lab: Notebook 1's
`reconcile_stuck_transactions` flips a timed-out transaction to `succeeded`
without writing any ledger entries. Let's find that gap by looking for it.

In [ ]:
import random

# ── Break type 1: succeeded transactions with no ledger entries ─────────
# This is a purely internal check — the ledger against our own transactions.
conn = get_db()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
cur.execute("""
    SELECT t.id, t.amount_cents, t.payment_intent_id
    FROM transactions t
    LEFT JOIN ledger_entries le ON le.transaction_id = t.id
    WHERE t.status = 'succeeded' AND t.type = 'charge' AND le.id IS NULL
    ORDER BY t.id
""")
unbooked = cur.fetchall()
cur.close(); conn.close()

print("🔎 Internal check — succeeded charges with NO ledger entries")
print("=" * 70)
if unbooked:
    total = sum(r["amount_cents"] for r in unbooked)
    for r in unbooked[:10]:
        print(f"   {r['id']:<20} ${r['amount_cents'] / 100:>10,.2f}  "
              f"(intent {r['payment_intent_id']})")
    if len(unbooked) > 10:
        print(f"   … and {len(unbooked) - 10} more")
    print(f"\n   💥 {len(unbooked)} charge(s), ${total / 100:,.2f}, money that moved")
    print("      and never hit the books. Debits still equal credits — the")
    print("      balance check cannot see this class of bug at all.")
else:
    print("   None found (run Notebook 1 first to create some).")


# ── Break types 2 and 3: our ledger vs the processor's settlement file ──
def fetch_settlement_file(seed=11):
    """Stand-in for 'download today's settlement report from the processor'.

    We build it FROM our own succeeded charges, then deliberately perturb it,
    because a settlement file that agrees perfectly teaches nothing.
    """
    rng = random.Random(seed)
    conn = get_db()
    cur = conn.cursor()
    cur.execute("""
        SELECT t.id, t.amount_cents FROM transactions t
        WHERE t.status = 'succeeded' AND t.type = 'charge'
        ORDER BY t.id LIMIT 12
    """)
    rows = cur.fetchall()
    cur.close(); conn.close()

    settlement = {}
    for i, (txn_id, amount) in enumerate(rows):
        if i == 1:
            continue                                    # they never settled this one
        if i == 2:
            settlement[txn_id] = amount - 30            # 30c processing fee deducted
        else:
            settlement[txn_id] = amount
    # …and one they settled that we have no record of at all.
    settlement["txn_ghost_0001"] = 1234
    return settlement


def ledger_amount_by_txn():
    """What OUR books say each transaction moved (sum of its debits)."""
    conn = get_db()
    cur = conn.cursor()
    cur.execute("""
        SELECT le.transaction_id,
               SUM(CASE WHEN le.entry_type = 'debit' THEN le.amount_cents ELSE 0 END)
        FROM ledger_entries le
        JOIN transactions t ON t.id = le.transaction_id
        WHERE t.type = 'charge'
        GROUP BY le.transaction_id
    """)
    result = {row[0]: row[1] for row in cur.fetchall()}
    cur.close(); conn.close()
    return result


def reconcile(settlement, ours):
    """Three-way diff. Returns breaks grouped by type."""
    breaks = {"missing_in_ledger": [], "missing_in_settlement": [], "amount_mismatch": []}
    for txn_id, their_amount in settlement.items():
        our_amount = ours.get(txn_id)
        if our_amount is None:
            breaks["missing_in_ledger"].append((txn_id, their_amount))
        elif our_amount != their_amount:
            breaks["amount_mismatch"].append((txn_id, our_amount, their_amount))
    for txn_id, our_amount in ours.items():
        if txn_id not in settlement:
            breaks["missing_in_settlement"].append((txn_id, our_amount))
    return breaks


settlement = fetch_settlement_file()
ours = ledger_amount_by_txn()
breaks = reconcile(settlement, ours)

print()
print("🔁 Reconciliation — our ledger vs the processor's settlement file")
print("=" * 70)
print(f"   Settlement lines: {len(settlement):>4}     Ledger transactions: {len(ours):>4}")
print()
for txn_id, amount in breaks["missing_in_ledger"]:
    print(f"   ❗ MISSING IN LEDGER      {txn_id:<20} they settled ${amount / 100:,.2f}")
    print(f"      → money left the customer and our books never recorded it.")
    print(f"        Investigate, then book it. Never 'adjust' the ledger to match.")
for txn_id, ours_amt, theirs in breaks["amount_mismatch"]:
    print(f"   ❗ AMOUNT MISMATCH        {txn_id:<20} "
          f"ours ${ours_amt / 100:,.2f} vs theirs ${theirs / 100:,.2f} "
          f"(Δ ${(ours_amt - theirs) / 100:,.2f})")
    print(f"      → almost always a fee we haven't modelled. The fix is a THIRD")
    print(f"        ledger account (processing_fees), not a smaller charge.")
for txn_id, amount in breaks["missing_in_settlement"][:5]:
    print(f"   ❗ MISSING IN SETTLEMENT  {txn_id:<20} we booked ${amount / 100:,.2f}")
    print(f"      → we recorded revenue the processor never sent. Often an")
    print(f"        authorisation booked as a capture.")
extra = len(breaks["missing_in_settlement"]) - 5
if extra > 0:
    print(f"   … and {extra} more missing-in-settlement break(s)")

total_breaks = sum(len(v) for v in breaks.values())
print()
print(f"   {total_breaks} break(s) for a human to work through.")
assert breaks["missing_in_ledger"], "expected the ghost settlement line to surface"
assert breaks["amount_mismatch"], "expected the fee-deduction mismatch to surface"

print()
print("💡 The rules that make reconciliation trustworthy:")
print("   1. The ledger is append-only. You resolve a break by writing a NEW")
print("      correcting entry, never by editing history.")
print("   2. Breaks are aged. A break open for 3 days is an incident; a break")
print("      open for 30 is a restatement.")
print("   3. Unexplained-break VALUE is the metric to alert on, not the count.")
print("      A hundred 1c rounding breaks matter less than one $50,000 break.")
print("   4. Reconcile against the party that HOLDS the money — the bank and")
print("      the processor — not against another one of your own services.")

---
## 9. Summary

| Concept | Key Point |
|---------|----------|
| **Double-Entry** | Every money movement creates two entries: a debit and a credit |
| **Balance Check** | Total debits must always equal total credits |
| **Atomicity** | Both entries are written in a single DB transaction — all or nothing |
| **Auditability** | The ledger is append-only — you never update or delete entries |
| **Error Detection** | If debits ≠ credits for any transaction, you've found a bug |
| **Enforcement** | A `DEFERRABLE INITIALLY DEFERRED` constraint trigger rejects an unbalanced transaction at `COMMIT` — detection after the fact is not enough |
| **Reconciliation** | Balanced books can still be wrong. Diff the ledger against the processor's settlement file daily and work the breaks |

### Why This Matters in System Design Interviews

When an interviewer asks "how do you ensure financial integrity?", the answer is:
1. **Double-entry bookkeeping** to track every dollar.
2. **Database transactions** to ensure atomic writes, plus a deferred constraint
   trigger so an unbalanced transaction cannot commit in the first place.
3. **Balance verification queries** to catch bugs — closing the books daily,
   because summing 1.4 trillion rows is not a nightly job.
4. **Append-only audit logs** for compliance; corrections are new entries.
5. **Daily reconciliation against the processor and the bank**, because an
   internally consistent ledger can still disagree with where the money is.

➡️  Next notebook: **Fraud Detection Basics**